# Spotimind — Phase 0: TRIBE v2 feasibility (audio-only, Colab T4)

Run top to bottom on a **T4 GPU** runtime. Cell P0.2 installs and then **kills the kernel to restart it** (expected); continue from P0.3.
No Hugging Face token is needed for audio-only (verified: only `facebook/tribev2` and `facebook/w2v-bert-2.0` get downloaded).

**Results recorded 2026-10-04** (tribev2 @ `af58661`, Python 3.13, torch 2.6.0, Tesla T4):

| Check | Result |
|---|---|
| Install | `uv pip install tribev2[plotting]@af58661`, then force `torch==2.6.0 torchaudio==2.6.0 numpy==2.2.6`, restart. The DataCamp `numpy<2.1` pin is outdated (the package pins numpy 2.2.6). |
| Audio-only call | Build an `Audio` event DataFrame → `get_audio_and_text_events(df, audio_only=True)` → `model.predict(events)`. Do **not** use `get_events_dataframe(audio_path=…)`: it runs WhisperX + the LLaMA text branch. |
| Extractors skipped | text + video are auto-removed ("no corresponding events"); missing modalities are fed as zeros. No gated model needed. |
| Output | `float32 [T, 20484]`, fsaverage5, left 10242 then right 10242 (L/R per-region contrast r=0.97 vs −0.10 with shuffled vertices). T ≈ ceil(duration), 1 Hz. |
| Lag | Already compensated by the model (offset 5 s). Auditory rise starts ~1–2 s before music onset, plateaus by ~+4 s; drop starts ~2 s before offset. Timeline is stimulus-aligned but smeared by ±2–4 s. |
| Transient | First ~2 s and last ~1–2 s of any input are edge-affected → proposed `TRIM_SECONDS = 3` at start, drop last 2 s. |
| Window boundary | Inference runs in 100 s windows without overlap: mild discontinuity at t=100, 200 … (frame diff ≈2.3× mean). 60 s audio-chunk boundaries are clean. |
| Speed / memory | **5.5 s compute per audio-minute**, peak 8.2 GB. 100 songs × 3.5 min ≈ **32 min** → full songs (cap 300 s) are affordable; the 90 s window isn't needed. |
| Sanity | Music vs silence: auditory core (Heschl / transverse sulcus / planum temporale) clearly up, ranks 7/11/16 of 75 Destrieux regions. **Lateral occipital regions rise even more** — likely a cross-modal prior learned from movies. Not noise, but a limitation to state. |
| Audio baseline | TRIBE's own cached w2v-bert extractor returns `[2 layer groups, 1024, T@2Hz]` → mean-pool → 2048-d. No extra model pass. |


In [ ]:
# [P0.1] Runtime check
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, platform; print(sys.version); print(platform.platform())
!free -h | head -2; nproc; df -h /content | tail -1

In [ ]:
# [P0.2] Install TRIBE v2 (pinned commit). Package pins numpy==2.2.6, torch==2.6 -> torchaudio must match (Colab ships 2.11).
# After this cell the runtime is restarted (os.kill) so the new numpy/torch are picked up.
!uv pip install --system -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git@af58661791a351a448a489042a28f6c37e1c14b7"
!uv pip install --system -q "torchaudio==2.6.0" "torch==2.6.0" "numpy==2.2.6"
!pip list 2>/dev/null | grep -iE "^(torch|torchaudio|torchvision|numpy|transformers|neuralset|tribev2|nilearn) "
import os; os.kill(os.getpid(), 9)  # restart runtime

In [ ]:
# [P0.3] Imports after restart + fetch a CC-licensed test song (Kevin MacLeod, CC BY 4.0)
import sys, numpy as np, torch, transformers, time, subprocess, json, os
from pathlib import Path
print("numpy", np.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
import tribev2, neuralset
from tribev2.demo_utils import TribeModel, get_audio_and_text_events
print("tribev2 import OK")

WORK = Path("/content/p0"); WORK.mkdir(exist_ok=True)
CACHE = WORK / "cache"
SONG_MP3 = WORK / "test_song.mp3"
URLS = ["https://incompetech.com/music/royalty-free/mp3-royaltyfree/Local%20Forecast%20-%20Elevator.mp3",
        "https://incompetech.com/music/royalty-free/mp3-royaltyfree/Sneaky%20Snitch.mp3"]
for u in URLS:
    if SONG_MP3.exists() and SONG_MP3.stat().st_size > 100_000: break
    subprocess.run(["wget", "-q", "-O", str(SONG_MP3), u])
dur = float(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(SONG_MP3)]))
print(f"test song: {SONG_MP3.stat().st_size/1e6:.1f} MB, {dur:.1f} s")
assert dur > 60

In [ ]:
# [P0.4] Load TRIBE v2 from HF -- deliberately WITHOUT an HF token, to test whether audio-only needs gated access.
import huggingface_hub
print("HF token present:", huggingface_hub.get_token() is not None)
t0 = time.time()
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=str(CACHE),
                                   config_update={"data.num_workers": 2})
print(f"loaded in {time.time()-t0:.1f}s | TR={model.data.TR}s | duration_trs={model.data.duration_trs} | features={model.data.features_to_use}")
print("neuro offset:", model.data.neuro.offset, "| mesh:", model.data.neuro.projection.mesh)
print(f"GPU mem allocated after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# [P0.5] Audio-only events. NOTE: model.get_events_dataframe(audio_path=...) runs WhisperX + text branch (LLaMA).
# For music we bypass it: build the Audio event ourselves and call get_audio_and_text_events(..., audio_only=True).
import pandas as pd

def to_wav(src, dst, start=None, dur=None):
    cmd = ["ffmpeg","-y","-v","error"] + (["-ss",str(start)] if start is not None else []) + ["-i",str(src)] \
          + (["-t",str(dur)] if dur is not None else []) + ["-ac","1","-ar","16000",str(dst)]
    subprocess.run(cmd, check=True); return dst

def audio_only_events(wav_path):
    ev = pd.DataFrame([{"type":"Audio","filepath":str(wav_path),"start":0,"timeline":"default","subject":"default"}])
    return get_audio_and_text_events(ev, audio_only=True)

# Onset test stimulus: 20 s silence + 40 s music + 20 s silence  -> measures lag/transient of the predictions
SONG_WAV = to_wav(SONG_MP3, WORK/"song.wav")
clip = WORK/"clip40.wav"; to_wav(SONG_MP3, clip, start=60, dur=40)
ONSET = WORK/"onset_test.wav"
subprocess.run(["ffmpeg","-y","-v","error","-f","lavfi","-t","20","-i","anullsrc=r=16000:cl=mono","-i",str(clip),
                "-f","lavfi","-t","20","-i","anullsrc=r=16000:cl=mono","-filter_complex","[0][1][2]concat=n=3:v=0:a=1",str(ONSET)], check=True)
ev_onset = audio_only_events(ONSET)
print(ev_onset[["type","start","duration","filepath"]].to_string())
print("event types:", ev_onset.type.unique().tolist())
assert set(ev_onset.type.unique()) == {"Audio"}, "expected only Audio events"

In [ ]:
# [P0.6] Predict on the onset test (also warms up / downloads w2v-bert). Check which extractor models got downloaded.
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
preds_on, segs_on = model.predict(events=ev_onset)
print(f"onset test: preds {preds_on.shape} {preds_on.dtype} in {time.time()-t0:.1f}s | peak GPU {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
print("first segment starts:", [round(s.start,2) for s in segs_on[:5]], "... last:", round(segs_on[-1].start,2))
hub = Path(huggingface_hub.constants.HF_HUB_CACHE)
print("HF models downloaded:", sorted(p.name for p in hub.glob("models--*")))
print("finite:", np.isfinite(preds_on).all(), "| mean/std:", preds_on.mean().round(4), preds_on.std().round(4))

In [ ]:
# [P0.7] Atlas on fsaverage5 (Destrieux, native fsaverage5 in nilearn). Assumes TRIBE vertex order = [left 10242, right 10242]
# (consistent with tribev2/utils_fmri.py which splits rec[:n] / rec[n:2n] as left/right; verified in P0.9a).
from nilearn import datasets
des = datasets.fetch_atlas_surf_destrieux()
labels = [l.decode() if isinstance(l, bytes) else str(l) for l in des["labels"]]
vlab = np.concatenate([des["map_left"], des["map_right"]]).astype(int)
assert vlab.shape[0] == preds_on.shape[1], (vlab.shape, preds_on.shape)
def roi_mask(names): idx = [labels.index(n) for n in names]; return np.isin(vlab, idx)
ROIS = {
  "auditory (Heschl+PT+transv sulcus)": ["G_temp_sup-G_T_transv","S_temporal_transverse","G_temp_sup-Plan_tempo"],
  "sup. temporal lateral":              ["G_temp_sup-Lateral"],
  "visual (occipital pole+cuneus+calcarine)": ["Pole_occipital","G_cuneus","S_calcarine"],
  "motor (precentral)":                 ["G_precentral"],
}
masks = {k: roi_mask(v) for k,v in ROIS.items()}
for k,m in masks.items(): print(f"{k:42s} {m.sum():5d} vertices")
def roi_ts(P): return {k: P[:, m].mean(1) for k,m in masks.items()}
ts = roi_ts(preds_on)
print("\nt(s)  " + " | ".join(k.split(' ')[0][:8] for k in ts))
for t in range(0, 80, 2):
    flag = "  <- music on" if t==20 else ("  <- music off" if t==60 else "")
    print(f"{t:4d}  " + " | ".join(f"{ts[k][t]:+.3f}" for k in ts) + flag)

In [ ]:
# [P0.8] Whole-cortex contrast: mean(music 24..56s) - mean(silence 4..16s and 64..76s), ranked by Destrieux region.
music_idx = np.arange(24, 57); sil_idx = np.r_[4:17, 64:77]
contrast = preds_on[music_idx].mean(0) - preds_on[sil_idx].mean(0)   # [20484]
rows = []
for li, name in enumerate(labels):
    m = vlab == li
    if m.sum() < 20: continue
    rows.append((name, m.sum(), contrast[m].mean()))
df_c = pd.DataFrame(rows, columns=["region","n_vert","music_minus_silence"]).sort_values("music_minus_silence", ascending=False)
print("TOP 10 regions (music > silence):"); print(df_c.head(10).to_string(index=False))
print("\nBOTTOM 5:"); print(df_c.tail(5).to_string(index=False))
aud_rank = df_c.reset_index(drop=True).index[df_c.reset_index(drop=True).region.isin(ROIS["auditory (Heschl+PT+transv sulcus)"])].tolist()
print("\nrank of auditory core regions (0=top) among", len(df_c), ":", aud_rank)
print("edge check: t0..2 =", ts["auditory (Heschl+PT+transv sulcus)"][:3].round(3), " last 3 =", ts["auditory (Heschl+PT+transv sulcus)"][-3:].round(3))
# Per-region split of the "visual" ROI (occipital pole goes UP with music, cuneus/calcarine slightly down)
for n in ROIS["visual (occipital pole+cuneus+calcarine)"]:
    mm = vlab == labels.index(n); print(f"  {n:16s} music-silence = {contrast[mm].mean():+.3f}")

In [ ]:
# [P0.9] (a) vertex-order sanity: L/R contrast maps should correlate (fsaverage5 L and R vertex indices are NOT homologous,
#            so compare per-region means instead).  (b) full-song timing + memory + 100-s window boundary check.
nL = 10242
regL = np.array([contrast[:nL][des["map_left"]==i].mean() if (des["map_left"]==i).sum()>10 else np.nan for i in range(len(labels))])
regR = np.array([contrast[nL:][des["map_right"]==i].mean() if (des["map_right"]==i).sum()>10 else np.nan for i in range(len(labels))])
ok = ~np.isnan(regL) & ~np.isnan(regR)
print(f"(a) L vs R per-region contrast correlation: r={np.corrcoef(regL[ok], regR[ok])[0,1]:.3f}  (high => vertex order/atlas alignment OK)")
rng = np.random.default_rng(0); perm = rng.permutation(nL)
regRp = np.array([contrast[nL:][perm][des["map_right"]==i].mean() if (des["map_right"]==i).sum()>10 else np.nan for i in range(len(labels))])
print(f"    control with shuffled right-hemi vertices: r={np.corrcoef(regL[ok], regRp[ok])[0,1]:.3f}")

# (b) full song
song_dur = float(subprocess.check_output(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(SONG_WAV)]))
ev_song = audio_only_events(SONG_WAV)
print("\n(b) song events:", len(ev_song), "rows;", ev_song[["start","duration"]].values.tolist())
torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); t0 = time.time()
preds_song, segs_song = model.predict(events=ev_song, verbose=False)
torch.cuda.synchronize(); el = time.time() - t0
peak = torch.cuda.max_memory_allocated()/1e9
print(f"song {song_dur:.1f}s -> preds {preds_song.shape} in {el:.1f}s  => {el/(song_dur/60):.1f} s compute per audio-minute | peak GPU {peak:.2f} GB")
a = preds_song[:, masks["auditory (Heschl+PT+transv sulcus)"]].mean(1)
d = np.abs(np.diff(preds_song, axis=0)).mean(1)   # mean abs frame-to-frame change over all vertices
print("mean |diff| overall:", d.mean().round(4), "| around t=100 (98..102):", d[97:102].round(4), "| around t=0..3:", d[:3].round(4))
print("auditory ts (every 5s):", a[::5].round(2))

In [ ]:
# [P0.10] (a) chunk boundaries at 60/120 s  (b) baseline audio features straight from TRIBE's own (cached) w2v-bert extractor
print("(a) |diff| at 58..62:", d[57:62].round(4), "| 118..122:", d[117:122].round(4), "| global p95:", np.percentile(d,95).round(4))
af = model.data.audio_feature
print("audio extractor:", type(af).__name__, af.model_name, "| layers", af.layers, "| layer_agg", af.layer_aggregation, "| freq", af.frequency)
t0 = time.time()
try:
    feats = af(ev_song[ev_song.type=="Audio"], start=0.0, duration=song_dur)
    feats = feats.detach().cpu().numpy() if hasattr(feats, "detach") else np.asarray(feats)
    print(f"(b) TRIBE w2v-bert features via extractor: shape {feats.shape} dtype {feats.dtype} in {time.time()-t0:.1f}s (cache hit expected)")
    src = "tribe_extractor"
except Exception as e:
    print("(b) extractor call FAILED:", repr(e)[:300]); feats = None; src = None
if feats is not None:
    # [n_layer_groups, D, T] or [D, T]; mean-pool over time
    pooled = feats.reshape(-1, feats.shape[-1]).mean(-1) if feats.ndim == 2 else feats.mean(-1).reshape(-1)
    print("pooled baseline vector:", pooled.shape, "finite", np.isfinite(pooled).all(), "| T axis ~ 2 Hz?", feats.shape[-1], "vs", round(song_dur*2))

In [ ]:
# [P0.11] Per-region time courses (song + onset test) and the Phase 0 checklist
import matplotlib.pyplot as plt
PLOT_ROIS = {"auditory core": ROIS["auditory (Heschl+PT+transv sulcus)"], "sup. temporal lat.": ["G_temp_sup-Lateral"],
             "occipital pole/lat.": ["Pole_occipital","S_occipital_ant","G_and_S_occipital_inf"], "precentral": ["G_precentral"],
             "mid. frontal": ["G_front_middle","S_front_middle"]}
pm = {k: roi_mask(v) for k,v in PLOT_ROIS.items()}
fig, ax = plt.subplots(2,1,figsize=(12,6))
for k,m in pm.items():
    ax[0].plot(preds_on[:,m].mean(1), label=k); ax[1].plot(preds_song[:,m].mean(1), label=k)
for x in (20,60): ax[0].axvline(x, c="k", ls="--", lw=.8)
ax[1].axvline(100, c="r", ls=":", lw=.8)
ax[0].set_title("Onset test: 20s silence | 40s music | 20s silence"); ax[1].set_title("Full song (red: 100-s inference window boundary)")
ax[0].legend(fontsize=8, ncol=5); ax[1].set_xlabel("seconds"); plt.tight_layout(); fig.savefig(WORK/"p0_roi_timecourses.png", dpi=110); plt.show()

summary = {
 "install": "uv pip install --system tribev2[plotting]@af58661 ; then torchaudio==2.6.0 torch==2.6.0 numpy==2.2.6 ; restart runtime. (No numpy<2.1 pin needed.)",
 "python": sys.version.split()[0], "gpu": torch.cuda.get_device_name(0),
 "audio_only_call": "get_audio_and_text_events(DataFrame([{type:'Audio',filepath,start:0,timeline,subject}]), audio_only=True) -> model.predict(events)",
 "hf_token_needed": False, "models_downloaded": sorted(p.name for p in hub.glob("models--*")),
 "output_shape_song": list(preds_song.shape), "n_vertices": int(preds_song.shape[1]), "space": "fsaverage5 (L 10242 then R 10242)",
 "tr_s": model.data.TR, "model_offset_s": model.data.neuro.offset, "inference_window_s": model.data.duration_trs * model.data.TR,
 "compute_s_per_audio_min": round(el/(song_dur/60),1), "peak_gpu_gb": round(peak,2),
 "est_100_songs_full_min": round(100*3.5*el/(song_dur/60)/60,1),
 "baseline_feature_shape": list(feats.shape), "baseline_source": src,
}
print(json.dumps(summary, indent=1))
json.dump(summary, open(WORK/"p0_summary.json","w"), indent=1)